# Production Architecture: Real-Time Arabic Sign Language Recognition
This notebook implements the production-ready pipeline for the SLR system.
It features:
1. **MediaPipe Hands**: Extracts hand landmarks (prepared for future temporal models).
2. **Improved Stabilization**: A robust `commit-once-then-wait` tracker to prevent repetitive letters.
3. **Arabic MLP Integration**: Hooks up the final Arabic `.h5` model.

In [14]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import time
from collections import deque
import os

print("✓ Libraries imported successfully")


✓ Libraries imported successfully


## 1. GPU Configuration
Ensure TensorFlow doesn't crash by managing GPU memory growth.

In [15]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU configured: {len(gpus)} GPU(s) available")
    except RuntimeError as e:
        print(f"⚠️ GPU config error: {e}")
else:
    print("ℹ️ No GPU detected, using CPU")


✓ GPU configured: 1 GPU(s) available


## 2. Configuration & Parameters
Set paths, thresholds, and class labels.

In [16]:
# Model Path (Arabic Final Model)
MLP_MODEL_PATH = r"arsl_mediapipe_mlp_model_final.h5"
MOBILENET_MODEL_PATH = r"mobilenet_arabic_final.h5"
DATASET_PATH = r"FINAL_CLEAN_DATASET.csv"

# Stabilization Settings
STABILIZATION_WINDOW_SIZE = 15  # Buffer size for majority vote
STABILIZATION_THRESHOLD = 11    # Minimum matches in buffer (11/15 = ~73% majority)
MIN_CONFIDENCE = 0.75           # Minimum confidence to consider a prediction
HOLD_COOLDOWN_SECONDS = 1.2     # Soft-lock time after committing a letter

# Display Settings
DISPLAY_WIDTH = 1280
DISPLAY_HEIGHT = 720

# Arabic Class Labels (28 Letters + 3 Controls)
CLASS_LABELS = [
    'ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر',
    'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف',
    'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ي',
    'space', 'del', 'nothing'
]
print("✓ Settings configured")


✓ Settings configured


## 3. Preprocessing (MediaPipe Hands)
Extracting hand landmarks. Using Holistic prepares the pipeline for future GRU models that require face/pose data.

In [17]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    max_num_hands=1
)

def extract_features(results):
    """
    Extracts hand features compatible with the current MLP model.
    Returns a flattened array of 63 features, or None if no hand detected.
    """
    # Prioritize right hand, then left hand
    if results.multi_hand_landmarks:
        landmarks = results.multi_hand_landmarks[0].landmark
    else:
        return None

    # Extract 21 points * 3 coordinates = 63 features
    features = np.array([[lm.x, lm.y, lm.z] for lm in landmarks]).flatten()
    return features.reshape(1, -1)

print("✓ Preprocessing functions defined")


✓ Preprocessing functions defined


## 4. Stabilization Tracker
This class manages the rolling buffer and enforces the `commit-once-then-wait` logic.

In [18]:
class StabilizationTracker:
    def __init__(self):
        self.buffer = deque(maxlen=STABILIZATION_WINDOW_SIZE)
        self.committed_label = None
        self.cooldown_until = 0

    def update(self, label, confidence):
        now = time.time()
        
        # If no hand/sign detected, clear buffer and release lock
        if label is None or label == 'nothing':
            self.buffer.clear()
            self.committed_label = None
            return None, 0.0, "waiting", 0

        # If in cooldown for this specific label, ignore it
        if self.committed_label == label and now < self.cooldown_until:
            return label, confidence, "cooldown", 100

        # Add to buffer
        self.buffer.append(label)
        
        # Calculate majority
        count = self.buffer.count(label)
        progress = (count / STABILIZATION_THRESHOLD) * 100
        
        if count >= STABILIZATION_THRESHOLD and len(self.buffer) == STABILIZATION_WINDOW_SIZE:
            # Commit the label
            self.committed_label = label
            self.cooldown_until = now + HOLD_COOLDOWN_SECONDS
            self.buffer.clear()
            return label, confidence, "committed", 100
            
        return label, confidence, "stabilizing", min(100, int(progress))

print("✓ Tracker class defined")


✓ Tracker class defined


## 5. Load Model
Loads the trained Arabic MLP model.

In [19]:
try:
    if os.path.exists(MLP_MODEL_PATH) and os.path.exists(MOBILENET_MODEL_PATH):
        mlp_model = tf.keras.models.load_model(MLP_MODEL_PATH)
        mobilenet_model = tf.keras.models.load_model(MOBILENET_MODEL_PATH)
        print("✓ Both Arabic MLP and MobileNet Models loaded successfully")
        print("✓ Arabic MLP Model loaded successfully")
    else:
        print(f"❌ Error: Model not found at {MLP_MODEL_PATH}")
        print("Please check the path and run this cell again.")
        model = None
except Exception as e:
    print(f"❌ Error loading model: {e}")
    model = None


Your GPU may run slowly with dtype policy mixed_float16 because it does not have compute capability of at least 7.0. Your GPU:
  NVIDIA GeForce MX150, compute capability 6.1
See https://developer.nvidia.com/cuda-gpus for a list of GPUs and their compute capabilities.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once
✓ Both Arabic MLP and MobileNet Models loaded successfully
✓ Arabic MLP Model loaded successfully


## 6. Arabic Text Display Utility
Handles rendering right-to-left Arabic text on OpenCV frames.

In [20]:
try:
    from PIL import Image, ImageDraw, ImageFont
    import arabic_reshaper
    from bidi.algorithm import get_display
    ARABIC_TEXT_OK = True
    print("✓ Arabic text rendering libraries found")
except ImportError:
    ARABIC_TEXT_OK = False
    print("⚠️ Arabic text libraries missing (pip install arabic-reshaper python-bidi). Using basic text.")

def draw_arabic_text(frame, text, position, font_size=40, color=(255, 255, 255)):
    if not ARABIC_TEXT_OK or not text:
        cv2.putText(frame, text, position, cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        return frame
        
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)
    
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil_image)
    
    # Try common font paths, fallback to default
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        font = ImageFont.load_default()
        
    rgb_color = (color[2], color[1], color[0])
    draw.text(position, bidi_text, fill=rgb_color, font=font)
    
    return cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)


⚠️ Arabic text libraries missing (pip install arabic-reshaper python-bidi). Using basic text.


## 7. Main Recognition Loop
Run this cell to open the webcam and start recognizing signs.

In [21]:
def run_sign_recognition():
    if mlp_model is None or mobilenet_model is None:
        print("❌ Model not loaded. Cannot run recognition.")
        return
        
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Cannot access camera")
        return
        
    print("\n==================================================")
    print("🤟 ARABIC SIGN LANGUAGE RECOGNITION STARTED")
    print("==================================================")
    print("Controls:")
    print("  'q' - Quit")
    print("  'c' - Clear sentence")
    print("==================================================\n")
    
    window_name = "Arabic SLR - Production Architecture"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, DISPLAY_WIDTH, DISPLAY_HEIGHT)
    
    tracker = StabilizationTracker()
    predicted_sentence = ""
    
    fps_start_time = time.time()
    frame_count = 0
    fps_display = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
                
            frame = cv2.resize(frame, (DISPLAY_WIDTH, DISPLAY_HEIGHT))
            
            # MediaPipe requires RGB
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            rgb_frame.flags.writeable = False
            results = hands.process(rgb_frame)
            rgb_frame.flags.writeable = True
            
            # Default UI state
            display_status = "No hand detected"
            status_color = (150, 150, 150)
            
            # Draw Holistic landmarks
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # Extract features and Predict
            features = extract_features(results)
            if features is not None:
                # Inference
                input_tensor = tf.cast(features, tf.float32)
                # 1. MediaPipe MLP Prediction (Landmarks)
                mlp_pred = mlp_model.predict(input_tensor, verbose=0)[0]

                # 2. MobileNetV2 Prediction (Image Crop)
                h, w = frame.shape[:2]
                hand_landmarks = results.multi_hand_landmarks[0]
                x_coords = [lm.x * w for lm in hand_landmarks.landmark]
                y_coords = [lm.y * h for lm in hand_landmarks.landmark]
                x1, y1 = max(0, int(min(x_coords)-50)), max(0, int(min(y_coords)-50))
                x2, y2 = min(w, int(max(x_coords)+50)), min(h, int(max(y_coords)+50))
                hand_crop = frame[y1:y2, x1:x2]

                if hand_crop.size > 0:
                    hand_img = cv2.resize(hand_crop, (224, 224))
                    hand_img = np.expand_dims(hand_img, axis=0) / 255.0
                    mob_pred = mobilenet_model.predict(hand_img, verbose=0)[0]
                    # Fusion
                    prediction = (mlp_pred * 0.6) + (mob_pred * 0.4)
                else:
                    prediction = mlp_pred
                
                class_idx = np.argmax(prediction)
                conf = float(prediction[class_idx])
                raw_label = CLASS_LABELS[class_idx]
                
                if conf < MIN_CONFIDENCE:
                    raw_label = None
                    display_status = f"Low confidence: {conf:.0%}"
                    status_color = (0, 100, 255)
                    tracker.update(None, 0.0) # Reset tracker
                else:
                    # Stabilize
                    tracked_label, tracked_conf, status, progress = tracker.update(raw_label, conf)
                    
                    if status == "stabilizing":
                        display_status = f"{tracked_label} ({tracked_conf:.0%}) Stabilizing {progress}%"
                        status_color = (0, 255, 255)
                    elif status == "cooldown":
                        display_status = f"{tracked_label} ({tracked_conf:.0%}) ✓ Committed - change sign"
                        status_color = (255, 200, 0)
                    elif status == "committed":
                        display_status = f"{tracked_label} ({tracked_conf:.0%}) ✓ COMMITTED!"
                        status_color = (0, 255, 0)
                        
                        # Sentence building
                        if tracked_label == "space":
                            if not predicted_sentence.endswith(" "):
                                predicted_sentence += " "
                        elif tracked_label == "del":
                            if predicted_sentence:
                                predicted_sentence = predicted_sentence[:-1]
                        elif tracked_label != "nothing":
                            predicted_sentence += tracked_label
            else:
                tracker.update(None, 0.0)
                
            # Mirror the frame
            frame = cv2.flip(frame, 1)
            
            # Calculate FPS
            frame_count += 1
            if time.time() - fps_start_time >= 1.0:
                fps_display = frame_count
                frame_count = 0
                fps_start_time = time.time()
                
            # Draw UI
            cv2.rectangle(frame, (0, 0), (DISPLAY_WIDTH, 70), (30, 30, 30), -1)
            cv2.putText(frame, f"FPS: {fps_display} | 'q'=quit 'c'=clear", (10, 25), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            
            # Draw status (handle Arabic correctly)
            frame = draw_arabic_text(frame, display_status, (10, 55), font_size=30, color=status_color)
            
            # Draw sentence bar
            bar_height = 80
            cv2.rectangle(frame, (0, DISPLAY_HEIGHT - bar_height), 
                         (DISPLAY_WIDTH, DISPLAY_HEIGHT), (30, 30, 30), -1)
            
            frame = draw_arabic_text(frame, predicted_sentence[-40:], (20, DISPLAY_HEIGHT - 30), 
                                   font_size=40, color=(255, 255, 255))
            
            cv2.imshow(window_name, frame)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('c'):
                predicted_sentence = ""
                tracker.update(None, 0.0)
                print("🗑️ Sentence cleared")
                
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print(f"\n📝 Final sentence: {predicted_sentence}")

# Run it!
run_sign_recognition()



🤟 ARABIC SIGN LANGUAGE RECOGNITION STARTED
Controls:
  'q' - Quit
  'c' - Clear sentence


📝 Final sentence: 


ValueError: in user code:

    File "c:\Users\adelg\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "c:\Users\adelg\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\adelg\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "c:\Users\adelg\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "c:\Users\adelg\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "c:\Users\adel